<div style="padding: 30px; background: linear-gradient(135deg, #2c3e50 0%, #3498db 100%); border-radius: 10px; margin: 10px 0;">
<h1 style="color: white; margin: 0; font-size: 2em;">PS6E4: Predicting Irrigation Need</h1>
<h3 style="color: #ecf0f1; margin: 10px 0 0 0; font-weight: normal;">End-to-End Pipeline with 3-Model Ensemble</h3>
<hr style="border-color: rgba(255,255,255,0.3); margin: 15px 0;">
<p style="color: #bdc3c7; margin: 0;">Competition: Playground Series S6E4 | Metric: Balanced Accuracy | GPU: P100</p>
</div>

<div style="padding: 15px 20px; background-color: #eef5fc; border-left: 4px solid #2980b9; border-radius: 4px; margin: 10px 0;">
<strong>Approach:</strong>

- Feature-engine pipeline with magic formula and domain features
- Three GBDT models: XGBoost, LightGBM, CatBoost
- Hill climbing ensemble with threshold optimization
- Trains end-to-end on Kaggle with P100 GPU
</div>

<div style="padding: 15px; background-color: #fef9e7; border-left: 4px solid #f39c12; border-radius: 4px; margin: 10px 0;">
<strong>Key Insight:</strong> The original 10K dataset was generated by a deterministic
</div>

<div style="padding: 20px; border-left: 5px solid #2980b9; background: linear-gradient(to right, #eef5fc, #ffffff); margin: 10px 0;">
<h2 style="color: #2c3e50; margin: 0;">1. Setup and Installs</h2>
<p style="color: #7f8c8d; margin: 5px 0 0 0;">Install and import all required libraries</p>
</div>

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "feature-engine"])

In [ ]:
import gc
import warnings
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import differential_evolution

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score, confusion_matrix
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_sample_weight

from feature_engine.encoding import CountFrequencyEncoder
from feature_engine.creation import RelativeFeatures, MathFeatures
from feature_engine.discretisation import (
    EqualFrequencyDiscretiser,
    EqualWidthDiscretiser,
    DecisionTreeDiscretiser,
)

warnings.filterwarnings("ignore")

# Seaborn theme
sns.set_theme(
    style="whitegrid",
    palette="muted",
    font_scale=1.1,
    rc={
        "figure.figsize": (12, 6),
        "axes.titlesize": 14,
        "axes.labelsize": 12,
    },
)

# Consistent color palette for irrigation classes
CLASS_COLORS = {"Low": "#27ae60", "Medium": "#f39c12", "High": "#e74c3c"}
CLASS_ORDER = ["Low", "Medium", "High"]
CLASS_PALETTE = [CLASS_COLORS[c] for c in CLASS_ORDER]

print("Setup complete.")

<div style="padding: 20px; border-left: 5px solid #2980b9; background: linear-gradient(to right, #eef5fc, #ffffff); margin: 10px 0;">
<h2 style="color: #2c3e50; margin: 0;">2. Load Data</h2>
<p style="color: #7f8c8d; margin: 5px 0 0 0;">Read competition CSVs and the original 10K dataset</p>
</div>

In [ ]:
KAGGLE_COMP_DIR = Path("/kaggle/input/competitions/playground-series-s6e4")
KAGGLE_ORIG_DIR = Path("/kaggle/input/datasets/wguesdon/ps6e4-irrigation-14-model-predictions")
ON_KAGGLE = Path("/kaggle").exists()

if ON_KAGGLE:
    # Debug: show what's mounted
    import os
    input_dir = Path("/kaggle/input")
    if input_dir.exists():
        print("Available inputs:", sorted(os.listdir(input_dir)))
    else:
        print("WARNING: /kaggle/input does not exist")
    DATA_DIR = KAGGLE_COMP_DIR
    ORIG_PATH = KAGGLE_ORIG_DIR / "irrigation_prediction.csv"
else:
    BASE_DIR = Path(__file__).resolve().parent.parent
    DATA_DIR = BASE_DIR / "data" / "raw"
    ORIG_PATH = DATA_DIR / "irrigation_prediction.csv"

print(f"DATA_DIR: {DATA_DIR} (exists: {DATA_DIR.exists()})")
print(f"ORIG_PATH: {ORIG_PATH} (exists: {ORIG_PATH.exists()})")

train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
orig = pd.read_csv(ORIG_PATH)

print(f"Train: {train.shape}")
print(f"Test:  {test.shape}")
print(f"Original: {orig.shape}")

<div style="padding: 20px; border-left: 5px solid #2980b9; background: linear-gradient(to right, #eef5fc, #ffffff); margin: 10px 0;">
<h2 style="color: #2c3e50; margin: 0;">3. Constants</h2>
<p style="color: #7f8c8d; margin: 5px 0 0 0;">Define column groups, constants, and encode target</p>
</div>

In [ ]:
TARGET = "Irrigation_Need"
LABEL_ORDER = ["High", "Low", "Medium"]  # LabelEncoder alphabetical order
SEED = 42

NUMERIC_COLS = [
    "Soil_pH", "Soil_Moisture", "Organic_Carbon", "Electrical_Conductivity",
    "Temperature_C", "Humidity", "Rainfall_mm", "Sunlight_Hours",
    "Wind_Speed_kmh", "Field_Area_hectare", "Previous_Irrigation_mm",
]

CAT_COLS = [
    "Soil_Type", "Crop_Type", "Crop_Growth_Stage", "Season",
    "Irrigation_Type", "Water_Source", "Mulching_Used", "Region",
]

BIN_COLS = [
    "Soil_Moisture", "Temperature_C", "Rainfall_mm",
    "Wind_Speed_kmh", "Humidity", "Previous_Irrigation_mm",
]

INTERACTION_PAIRS = [
    ("Crop_Growth_Stage", "Crop_Type"),
    ("Crop_Growth_Stage", "Irrigation_Type"),
    ("Irrigation_Type", "Season"),
    ("Crop_Type", "Season"),
    ("Mulching_Used", "Irrigation_Type"),
    ("Soil_Type", "Crop_Type"),
    ("Region", "Season"),
    ("Irrigation_Type", "Water_Source"),
]

# Encode target
le = LabelEncoder()
le.fit(LABEL_ORDER)
y_all = le.transform(train[TARGET])
y_orig = le.transform(orig[TARGET])
test_ids = test["id"]

print(f"Classes: {le.classes_}")
print(f"Target distribution:\n{train[TARGET].value_counts().to_string()}")

<div style="padding: 20px; border-left: 5px solid #27ae60; background: linear-gradient(to right, #eafaf1, #ffffff); margin: 10px 0;">
<h2 style="color: #2c3e50; margin: 0;">4. Exploratory Data Analysis</h2>
<p style="color: #7f8c8d; margin: 5px 0 0 0;">Visualize target distribution, feature distributions, and key patterns</p>
</div>

<div style="padding: 15px; background-color: #eafaf1; border-left: 4px solid #27ae60; border-radius: 4px; margin: 10px 0;">
The dataset has 630K synthetic training rows generated from a 10K original dataset.
The target has 3 classes with significant imbalance.
</div>

<div style="padding: 12px 20px; border-left: 4px solid #95a5a6; background-color: #f8f9fa; margin: 10px 0;">
<h3 style="color: #2c3e50; margin: 0;">4.1 Target Distribution</h3>
</div>

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
counts = train[TARGET].value_counts()
pcts = counts / len(train) * 100

bars = sns.barplot(
    x=pcts[CLASS_ORDER].values, y=CLASS_ORDER,
    palette=CLASS_PALETTE, ax=ax, edgecolor="white", linewidth=1.5,
)
for i, cls in enumerate(CLASS_ORDER):
    ax.text(
        pcts[cls] + 0.8, i, f"{counts[cls]:,}  ({pcts[cls]:.1f}%)",
        va="center", fontweight="bold", fontsize=11,
    )

ax.set_xlabel("Percentage of Training Samples")
ax.set_title("Target Distribution: Irrigation Need", fontweight="bold")
ax.set_xlim(0, 70)
sns.despine(left=True)
plt.tight_layout()
plt.show()

<div style="padding: 15px; background-color: #eafaf1; border-left: 4px solid #27ae60; border-radius: 4px; margin: 10px 0;">
The High class represents only ~3% of samples. Balanced accuracy gives equal
weight to each class. Correctly predicting the rare High class is critical.
</div>

<div style="padding: 12px 20px; border-left: 4px solid #95a5a6; background-color: #f8f9fa; margin: 10px 0;">
<h3 style="color: #2c3e50; margin: 0;">4.2 Top Feature Distributions by Class</h3>
</div>

In [ ]:
top_features = ["Soil_Moisture", "Wind_Speed_kmh", "Temperature_C", "Rainfall_mm"]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, col in enumerate(top_features):
    ax = axes[i]
    sns.violinplot(
        data=train, x=TARGET, y=col, order=CLASS_ORDER,
        palette=CLASS_COLORS, ax=ax, inner="quartile", linewidth=1,
        saturation=0.8, cut=0,
    )
    ax.set_title(col, fontweight="bold", fontsize=12)
    ax.set_xlabel("")

fig.suptitle("Top Feature Distributions by Class", fontsize=15, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

<div style="padding: 15px; background-color: #eafaf1; border-left: 4px solid #27ae60; border-radius: 4px; margin: 10px 0;">
**Observations:**
- Soil moisture is the strongest separator. Low moisture strongly predicts High irrigation need.
- Wind speed and temperature also show clear class separation.
- These are the features the magic formula relies on.
</div>

<div style="padding: 12px 20px; border-left: 4px solid #95a5a6; background-color: #f8f9fa; margin: 10px 0;">
<h3 style="color: #2c3e50; margin: 0;">4.3 Correlation Heatmap</h3>
</div>

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
corr = train[NUMERIC_COLS].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

sns.heatmap(
    corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r",
    center=0, vmin=-1, vmax=1, ax=ax, linewidths=0.5,
    cbar_kws={"shrink": 0.8, "label": "Pearson r"},
)
ax.set_title("Feature Correlation Matrix", fontweight="bold", fontsize=14)
plt.tight_layout()
plt.show()

<div style="padding: 15px; background-color: #eafaf1; border-left: 4px solid #27ae60; border-radius: 4px; margin: 10px 0;">
Most features are weakly correlated. This is favorable for building diverse models.
</div>

<div style="padding: 12px 20px; border-left: 4px solid #95a5a6; background-color: #f8f9fa; margin: 10px 0;">
<h3 style="color: #2c3e50; margin: 0;">4.4 Magic Formula Score by Class</h3>
</div>

<div style="padding: 15px 20px; background-color: #eef5fc; border-left: 4px solid #2980b9; border-radius: 4px; margin: 10px 0;">
The original 10K dataset was generated by a deterministic formula:

```
high_score = (Soil_Moisture < 25)*2 + (Rainfall_mm < 300)*2 + (Temperature > 30) + (Wind > 10)
low_score  = (Growth_Stage == Harvest)*2 + (Growth_Stage == Sowing)*2 + (Mulching == Yes)
magic_score = high_score - low_score
```

On the original data this achieves BA = 1.0.
On the noisy synthetic data the formula is not perfect but provides strong signal.
</div>

In [ ]:
magic_score = (
    (train["Soil_Moisture"] < 25).astype(int) * 2
    + (train["Rainfall_mm"] < 300).astype(int) * 2
    + (train["Temperature_C"] > 30).astype(int)
    + (train["Wind_Speed_kmh"] > 10).astype(int)
    - (train["Crop_Growth_Stage"] == "Harvest").astype(int) * 2
    - (train["Crop_Growth_Stage"] == "Sowing").astype(int) * 2
    - (train["Mulching_Used"] == "Yes").astype(int)
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: distribution by class
for cls in CLASS_ORDER:
    m = train[TARGET] == cls
    sns.kdeplot(
        magic_score[m], ax=axes[0], label=cls, color=CLASS_COLORS[cls],
        fill=True, alpha=0.2, linewidth=1.5,
    )
axes[0].set_title("Magic Score Distribution (Synthetic Data)", fontweight="bold")
axes[0].set_xlabel("Magic Score")
axes[0].legend(title="Class")

# Right: class proportions per score
score_class = pd.crosstab(magic_score, train[TARGET], normalize="index")
score_class = score_class[CLASS_ORDER]
score_class.plot(
    kind="bar", stacked=True, color=CLASS_PALETTE, ax=axes[1],
    edgecolor="white", linewidth=0.5,
)
axes[1].set_title("Class Proportions per Magic Score", fontweight="bold")
axes[1].set_xlabel("Magic Score")
axes[1].set_ylabel("Proportion")
axes[1].legend(title="Class", fontsize=9)
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)

plt.tight_layout()
plt.show()

magic_pred = np.where(magic_score <= 0, "Low", np.where(magic_score >= 4, "High", "Medium"))
magic_ba = balanced_accuracy_score(train[TARGET], magic_pred)
print(f"Magic formula BA on synthetic data: {magic_ba:.4f}")

<div style="padding: 12px 20px; border-left: 4px solid #95a5a6; background-color: #f8f9fa; margin: 10px 0;">
<h3 style="color: #2c3e50; margin: 0;">4.5 Categorical Feature Distribution by Class</h3>
</div>

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
for i, col in enumerate(CAT_COLS):
    ax = axes[i // 4, i % 4]
    ct = pd.crosstab(train[col], train[TARGET], normalize='index')[CLASS_ORDER]
    ct.plot(kind='bar', stacked=True, color=CLASS_PALETTE, ax=ax, legend=False, edgecolor='white', linewidth=0.5)
    ax.set_title(col, fontweight='bold', fontsize=11)
    ax.set_xlabel('')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=9)
    if i == 0:
        ax.legend(CLASS_ORDER, title='Class', fontsize=8, loc='upper right')
fig.suptitle('Class Proportions by Categorical Feature', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

<div style="padding: 15px; background-color: #eafaf1; border-left: 4px solid #27ae60; border-radius: 4px; margin: 10px 0;">
<strong>Key Finding:</strong> Crop Growth Stage is the strongest categorical separator. Harvest and Sowing stages have very different class proportions compared to other stages. Mulching also shows clear differentiation, consistent with the magic formula.
</div>

<div style="padding: 12px 20px; border-left: 4px solid #95a5a6; background-color: #f8f9fa; margin: 10px 0;">
<h3 style="color: #2c3e50; margin: 0;">4.6 Pairwise Scatter of Top 3 Numeric Features</h3>
</div>

In [ ]:
top3 = ['Soil_Moisture', 'Temperature_C', 'Rainfall_mm']
sample = train.sample(10000, random_state=42)
g = sns.pairplot(
    sample, vars=top3, hue=TARGET, hue_order=CLASS_ORDER,
    palette=CLASS_COLORS, diag_kind='kde', plot_kws={'alpha': 0.3, 's': 10},
    height=3,
)
g.figure.suptitle('Pairwise Feature Scatter (10K Sample)', fontweight='bold', y=1.02)
plt.show()

<div style="padding: 15px; background-color: #eafaf1; border-left: 4px solid #27ae60; border-radius: 4px; margin: 10px 0;">
<strong>Key Finding:</strong> Soil Moisture provides the clearest separation in scatter space. High irrigation samples cluster at low soil moisture values. Temperature and Rainfall show overlapping distributions but still contribute to separation when combined.
</div>

<div style="padding: 20px; border-left: 5px solid #f39c12; background: linear-gradient(to right, #fef9e7, #ffffff); margin: 10px 0;">
<h2 style="color: #2c3e50; margin: 0;">5. Feature Engineering</h2>
<p style="color: #7f8c8d; margin: 5px 0 0 0;">Seven-stage pipeline: magic formula, domain, feature-engine, TE priors, interactions</p>
</div>

<div style="padding: 15px 20px; background-color: #eef5fc; border-left: 4px solid #2980b9; border-radius: 4px; margin: 10px 0;">
<p style="margin: 0 0 10px 0;">The pipeline has seven stages:</p>
<ol style="margin: 0; padding-left: 20px;">
<li>Magic formula features (threshold flags, composite scores, boundary distances)</li>
<li>Domain features (water balance, heat stress, ET proxy, drying index)</li>
<li>Label encode categoricals (fitted on combined train+test+orig)</li>
<li>Feature-engine static transforms (frequency encoding, binning, relative features, math features)</li>
<li>Original TE priors (mean target per category from 10K original data, leakage free)</li>
<li>2-way interaction columns (all C(19,2) pairs, factorized, drop >50% unique)</li>
<li>In-fold: DecisionTreeDiscretiser + manual mean target encoding for top interaction pairs</li>
</ol>
</div>

In [ ]:
def add_magic_features(df):
    """Add magic formula features from the original dataset rule.

    The original 10K dataset was generated by a deterministic formula
    involving threshold comparisons. These features expose that structure.

    Args:
        df: DataFrame with raw feature columns.

    Returns:
        DataFrame with magic features appended.
    """
    df = df.copy()
    df["magic_soil_dry"] = (df["Soil_Moisture"] < 25).astype(np.int8)
    df["magic_rain_low"] = (df["Rainfall_mm"] < 300).astype(np.int8)
    df["magic_temp_hot"] = (df["Temperature_C"] > 30).astype(np.int8)
    df["magic_wind_high"] = (df["Wind_Speed_kmh"] > 10).astype(np.int8)
    df["magic_harvest"] = (df["Crop_Growth_Stage"] == "Harvest").astype(np.int8)
    df["magic_sowing"] = (df["Crop_Growth_Stage"] == "Sowing").astype(np.int8)
    df["magic_mulch_yes"] = (df["Mulching_Used"] == "Yes").astype(np.int8)

    df["magic_high_score"] = (
        df["magic_soil_dry"] * 2 + df["magic_rain_low"] * 2
        + df["magic_temp_hot"] + df["magic_wind_high"]
    )
    df["magic_low_score"] = (
        df["magic_harvest"] * 2 + df["magic_sowing"] * 2 + df["magic_mulch_yes"]
    )
    df["magic_score"] = df["magic_high_score"] - df["magic_low_score"]

    df["magic_soil_margin"] = 25 - df["Soil_Moisture"]
    df["magic_rain_margin"] = 300 - df["Rainfall_mm"]
    df["magic_temp_margin"] = df["Temperature_C"] - 30
    df["magic_wind_margin"] = df["Wind_Speed_kmh"] - 10
    df["magic_boundary_dist"] = df[
        ["magic_soil_margin", "magic_rain_margin",
         "magic_temp_margin", "magic_wind_margin"]
    ].abs().min(axis=1)
    return df


def add_domain_features(df):
    """Add domain-knowledge interaction features.

    Physics-inspired features for water balance, heat stress,
    evapotranspiration, and soil efficiency.

    Args:
        df: DataFrame with raw feature columns.

    Returns:
        DataFrame with domain features appended.
    """
    df = df.copy()
    df["water_balance"] = df["Rainfall_mm"] - df["Previous_Irrigation_mm"]
    df["heat_stress"] = df["Temperature_C"] * (100 - df["Humidity"]) / 100
    df["ET_proxy"] = (
        df["Temperature_C"] * df["Wind_Speed_kmh"]
        * (100 - df["Humidity"]) / 100
    )
    df["drying_index"] = (
        df["Wind_Speed_kmh"] * df["Temperature_C"]
        / df["Humidity"].clip(lower=1)
    )
    df["moisture_deficit"] = df["Soil_Moisture"] - df["Rainfall_mm"] / 10
    df["water_per_hectare"] = (
        (df["Rainfall_mm"] + df["Previous_Irrigation_mm"])
        / (df["Field_Area_hectare"] + 0.01)
    )
    df["moisture_temp_ratio"] = df["Soil_Moisture"] / (df["Temperature_C"] + 1)
    return df


def label_encode_categoricals(train_df, test_df, orig_df):
    """Label-encode categorical columns fitted on combined data.

    Args:
        train_df: Training DataFrame.
        test_df: Test DataFrame.
        orig_df: Original 10K DataFrame.

    Returns:
        Tuple of (train_df, test_df, orig_df) with _le columns added.
    """
    combined = pd.concat(
        [train_df[CAT_COLS], test_df[CAT_COLS], orig_df[CAT_COLS]],
        ignore_index=True,
    )
    for col in CAT_COLS:
        enc = LabelEncoder()
        enc.fit(combined[col].astype(str))
        for df in [train_df, test_df, orig_df]:
            df[f"{col}_le"] = enc.transform(df[col].astype(str))
    return train_df, test_df, orig_df


def apply_feature_engine_static(train_df, test_df):
    """Apply feature-engine transformers that do not use the target.

    Fit on train, transform both train and test. Uses:
    - CountFrequencyEncoder: category frequency as numeric.
    - EqualFrequencyDiscretiser: quantile binning (10 bins).
    - EqualWidthDiscretiser: fixed-width binning (15 bins).
    - RelativeFeatures: pairwise sub and mul.
    - MathFeatures: aggregate statistics (mean, std).

    Args:
        train_df: Training DataFrame.
        test_df: Test DataFrame.

    Returns:
        Tuple of (train_df, test_df, list of new column names).
    """
    new_cols = []

    # 1. Frequency encoding for categoricals
    cfe = CountFrequencyEncoder(
        encoding_method="frequency", variables=CAT_COLS, missing_values="ignore",
    )
    cfe.fit(train_df[CAT_COLS])
    cfe_train = cfe.transform(train_df[CAT_COLS].copy())
    cfe_test = cfe.transform(test_df[CAT_COLS].copy())
    for col in CAT_COLS:
        name = f"cfe_{col}"
        train_df[name] = cfe_train[col].values
        test_df[name] = cfe_test[col].values
        new_cols.append(name)

    # 2. Equal-frequency binning (quantile, 10 bins)
    efd = EqualFrequencyDiscretiser(
        q=10, variables=BIN_COLS, return_boundaries=False,
    )
    efd.fit(train_df[BIN_COLS])
    efd_train = efd.transform(train_df[BIN_COLS].copy())
    efd_test = efd.transform(test_df[BIN_COLS].copy())
    for col in BIN_COLS:
        name = f"efd10_{col}"
        train_df[name] = efd_train[col].values
        test_df[name] = efd_test[col].values
        new_cols.append(name)

    # 3. Equal-width binning (15 bins)
    ewd = EqualWidthDiscretiser(
        bins=15, variables=BIN_COLS, return_boundaries=False,
    )
    ewd.fit(train_df[BIN_COLS])
    ewd_train = ewd.transform(train_df[BIN_COLS].copy())
    ewd_test = ewd.transform(test_df[BIN_COLS].copy())
    for col in BIN_COLS:
        name = f"ewd15_{col}"
        train_df[name] = ewd_train[col].values
        test_df[name] = ewd_test[col].values
        new_cols.append(name)

    # 4. Relative features: pairwise differences and products
    rel_vars = ["Soil_Moisture", "Temperature_C", "Humidity"]
    rel_refs = ["Wind_Speed_kmh", "Rainfall_mm"]
    rel_input_cols = rel_vars + rel_refs
    rel_tf = RelativeFeatures(
        variables=rel_vars, reference=rel_refs, func=["sub", "mul"],
    )
    rel_tf.fit(train_df[rel_input_cols])
    rel_train = rel_tf.transform(train_df[rel_input_cols].copy())
    rel_test = rel_tf.transform(test_df[rel_input_cols].copy())
    rel_new = [c for c in rel_train.columns if c not in rel_input_cols]
    for col in rel_new:
        train_df[col] = rel_train[col].values
        test_df[col] = rel_test[col].values
    new_cols.extend(rel_new)

    # 5. Aggregate math features: mean and std of top features
    math_vars = ["Soil_Moisture", "Temperature_C", "Humidity", "Wind_Speed_kmh"]
    math_tf = MathFeatures(
        variables=math_vars, func=["mean", "std"], missing_values="ignore",
    )
    math_tf.fit(train_df[math_vars])
    math_train = math_tf.transform(train_df[math_vars].copy())
    math_test = math_tf.transform(test_df[math_vars].copy())
    math_new = [c for c in math_train.columns if c not in math_vars]
    for col in math_new:
        train_df[col] = math_train[col].values
        test_df[col] = math_test[col].values
    new_cols.extend(math_new)

    return train_df, test_df, new_cols


def add_original_te_priors(train_df, test_df, orig_df):
    """Add target encoding priors from the original 10K dataset.

    For each categorical column, compute mean target per category from
    the original data. This is leakage free because the original dataset
    is independent of the synthetic split.

    Args:
        train_df: Training DataFrame.
        test_df: Test DataFrame.
        orig_df: Original DataFrame with TARGET column.

    Returns:
        Tuple of (train_df, test_df, list of new column names).
    """
    target_enc = LabelEncoder()
    target_enc.fit(LABEL_ORDER)
    orig_y_num = target_enc.transform(orig_df[TARGET])
    global_mean = orig_y_num.mean()

    prior_cols = []
    for col in CAT_COLS:
        name = f"orig_prior_{col}"
        te_map = pd.Series(orig_y_num).groupby(orig_df[col]).mean()
        train_df[name] = train_df[col].map(te_map).fillna(global_mean).astype(np.float32)
        test_df[name] = test_df[col].map(te_map).fillna(global_mean).astype(np.float32)
        prior_cols.append(name)

    return train_df, test_df, prior_cols


def create_2way_interactions(train_df, test_df):
    """Create 2-way categorical interaction columns as factorized integers.

    Generates all C(19,2) = 171 pairs from numeric + categorical columns.
    Drops high-cardinality pairs where more than 50% of values are unique.

    Args:
        train_df: Training DataFrame.
        test_df: Test DataFrame.

    Returns:
        Tuple of (train_df, test_df, list of interaction column names).
    """
    all_cols = NUMERIC_COLS + CAT_COLS
    interaction_cols = []

    for col_a, col_b in combinations(all_cols, 2):
        name = f"{col_a}-{col_b}"
        combined = pd.concat([
            train_df[col_a].astype(str) + "_" + train_df[col_b].astype(str),
            test_df[col_a].astype(str) + "_" + test_df[col_b].astype(str),
        ], ignore_index=True)
        encoded, _ = combined.factorize()
        if pd.Series(encoded).nunique() > len(combined) // 2:
            continue
        train_df[name] = encoded[:len(train_df)]
        test_df[name] = encoded[len(train_df):]
        interaction_cols.append(name)

    return train_df, test_df, interaction_cols


def apply_fold_features(X_tr, X_va, X_te, y_tr):
    """Apply target-dependent feature-engine transformers inside a CV fold.

    DecisionTreeDiscretiser: supervised binning using a shallow decision tree.
    Manual mean target encoding for bigram interaction columns.
    Fit on training fold only, transform val and test.

    Args:
        X_tr: Training fold features.
        X_va: Validation fold features.
        X_te: Test features.
        y_tr: Training fold target (numeric).

    Returns:
        Tuple of (X_tr, X_va, X_te) with fold features added.
    """
    y_series = pd.Series(y_tr, index=X_tr.index)

    # 1. Decision tree supervised binning
    bin_cols_present = [c for c in BIN_COLS if c in X_tr.columns]
    dtd = DecisionTreeDiscretiser(
        variables=bin_cols_present,
        regression=False,
        scoring="balanced_accuracy",
        cv=3,
        param_grid={"max_depth": [2, 3]},
    )
    dtd.fit(X_tr[bin_cols_present], y_series)
    dtd_train = dtd.transform(X_tr[bin_cols_present].copy())
    dtd_val = dtd.transform(X_va[bin_cols_present].copy())
    dtd_test = dtd.transform(X_te[bin_cols_present].copy())
    for col in bin_cols_present:
        name = f"dtd_{col}"
        X_tr[name] = dtd_train[col].values
        X_va[name] = dtd_val[col].values
        X_te[name] = dtd_test[col].values

    # 2. Mean target encoding for interaction pairs
    interaction_present = []
    for c1, c2 in INTERACTION_PAIRS:
        le_c1 = f"{c1}_le" if f"{c1}_le" in X_tr.columns else c1
        le_c2 = f"{c2}_le" if f"{c2}_le" in X_tr.columns else c2
        name = f"bigram_{c1}_{c2}"
        for df in [X_tr, X_va, X_te]:
            df[name] = df[le_c1].astype(str) + "_" + df[le_c2].astype(str)
        interaction_present.append(name)

    global_mean = y_tr.mean()
    for col in interaction_present:
        te_map = y_series.groupby(X_tr[col]).mean()
        X_tr[f"{col}_te"] = X_tr[col].map(te_map).fillna(global_mean)
        X_va[f"{col}_te"] = X_va[col].map(te_map).fillna(global_mean)
        X_te[f"{col}_te"] = X_te[col].map(te_map).fillna(global_mean)

    # Drop the raw string bigram columns
    for col in interaction_present:
        X_tr.drop(columns=[col], inplace=True)
        X_va.drop(columns=[col], inplace=True)
        X_te.drop(columns=[col], inplace=True)

    return X_tr, X_va, X_te


def build_all_static_features(train_df, test_df, orig_df):
    """Full static feature engineering pipeline.

    Args:
        train_df: Training DataFrame.
        test_df: Test DataFrame.
        orig_df: Original 10K DataFrame.

    Returns:
        Tuple of (train_df, test_df, orig_df, feature_cols, interaction_cols).
    """
    print("[1/6] Magic formula features...")
    train_df = add_magic_features(train_df)
    test_df = add_magic_features(test_df)
    orig_df = add_magic_features(orig_df)

    print("[2/6] Domain features...")
    train_df = add_domain_features(train_df)
    test_df = add_domain_features(test_df)
    orig_df = add_domain_features(orig_df)

    print("[3/6] Label encoding...")
    train_df, test_df, orig_df = label_encode_categoricals(train_df, test_df, orig_df)

    print("[4/6] Feature-engine static transforms...")
    train_df, test_df, fe_cols = apply_feature_engine_static(train_df, test_df)
    print(f"       Feature-engine added {len(fe_cols)} columns")

    print("[5/6] Original TE priors...")
    train_df, test_df, prior_cols = add_original_te_priors(train_df, test_df, orig_df)
    print(f"       Original priors: {len(prior_cols)} columns")

    print("[6/6] 2-way interactions...")
    train_df, test_df, interaction_cols = create_2way_interactions(train_df, test_df)
    print(f"       Interaction columns: {len(interaction_cols)}")

    # Fill missing columns in orig for injection
    for col in train_df.columns:
        if col not in orig_df.columns:
            orig_df[col] = 0

    exclude = {"id", TARGET} | set(CAT_COLS)
    feature_cols = [c for c in train_df.columns if c not in exclude]
    print(f"Total features: {len(feature_cols)}")

    return train_df, test_df, orig_df, feature_cols, interaction_cols

In [ ]:
print("Running feature engineering pipeline...")
train, test, orig, feature_cols, interaction_cols = build_all_static_features(
    train, test, orig,
)
print("Feature engineering complete.")

<div style="padding: 20px; border-left: 5px solid #e74c3c; background: linear-gradient(to right, #fdedec, #ffffff); margin: 10px 0;">
<h2 style="color: #2c3e50; margin: 0;">6. Model Training (5-Fold CV)</h2>
<p style="color: #7f8c8d; margin: 5px 0 0 0;">XGBoost, LightGBM, CatBoost with 5-fold CV and original data injection</p>
</div>

<div style="padding: 15px 20px; background-color: #eef5fc; border-left: 4px solid #2980b9; border-radius: 4px; margin: 10px 0;">
Three GBDT models trained with fixed hyperparameters.
Each fold injects the original 10K rows into the training set.
In-fold target encoding is applied via DecisionTreeDiscretiser and manual mean encoding.

| Model | Key Parameters | Notes |
|-------|---------------|-------|
| XGBoost | depth=6, lr=0.01, 10K rounds, balanced weights | GPU hist |
| LightGBM | leaves=77, lr=0.01, is_unbalance=True | CPU |
| CatBoost | depth=6, lr=0.04, auto balanced weights | GPU |
</div>

In [ ]:
N_FOLDS = 5
N_ESTIMATORS = 10000
EARLY_STOP = 100


def train_xgb_fold(X_tr, y_tr, X_va, y_va):
    """Train XGBoost on one fold with balanced sample weights.

    Args:
        X_tr: Training features.
        y_tr: Training target.
        X_va: Validation features.
        y_va: Validation target.

    Returns:
        Fitted XGBClassifier.
    """
    from xgboost import XGBClassifier

    model = XGBClassifier(
        objective="multi:softprob",
        num_class=3,
        eval_metric="mlogloss",
        max_depth=6,
        learning_rate=0.01,
        n_estimators=N_ESTIMATORS,
        subsample=0.8,
        colsample_bytree=0.8,
        max_bin=1024,
        tree_method="hist",
        device="cuda",
        random_state=SEED,
        n_jobs=-1,
        verbosity=0,
    )
    weights = compute_sample_weight("balanced", y_tr)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        sample_weight=weights,
        verbose=False,
    )
    return model


def train_lgb_fold(X_tr, y_tr, X_va, y_va):
    """Train LightGBM on one fold with is_unbalance.

    Args:
        X_tr: Training features.
        y_tr: Training target.
        X_va: Validation features.
        y_va: Validation target.

    Returns:
        Fitted LGBMClassifier.
    """
    import lightgbm as lgb

    model = lgb.LGBMClassifier(
        objective="multiclass",
        num_class=3,
        num_leaves=77,
        learning_rate=0.01,
        n_estimators=N_ESTIMATORS,
        subsample=0.8,
        colsample_bytree=0.7,
        reg_lambda=7.0,
        is_unbalance=True,
        random_state=SEED,
        n_jobs=-1,
        verbosity=-1,
    )
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        callbacks=[
            lgb.early_stopping(EARLY_STOP, verbose=False),
            lgb.log_evaluation(0),
        ],
    )
    return model


def train_cat_fold(X_tr, y_tr, X_va, y_va):
    """Train CatBoost on one fold with balanced weights on GPU.

    Args:
        X_tr: Training features.
        y_tr: Training target.
        X_va: Validation features.
        y_va: Validation target.

    Returns:
        Fitted CatBoostClassifier.
    """
    from catboost import CatBoostClassifier

    model = CatBoostClassifier(
        loss_function="MultiClass",
        classes_count=3,
        auto_class_weights="Balanced",
        depth=6,
        learning_rate=0.04,
        iterations=N_ESTIMATORS,
        l2_leaf_reg=0.5,
        random_strength=8.0,
        bagging_temperature=0.4,
        task_type="GPU",
        random_seed=SEED,
        verbose=0,
    )
    model.fit(X_tr, y_tr, eval_set=(X_va, y_va), early_stopping_rounds=EARLY_STOP)
    return model


def run_model_cv(model_name, train_fn, X, y, X_test_df, X_orig_df, y_orig_arr,
                 feat_cols):
    """Run full 5-fold CV for one model.

    Args:
        model_name: Short name for display.
        train_fn: Callable(X_tr, y_tr, X_va, y_va) returning fitted model.
        X: Full training DataFrame.
        y: Full training target array.
        X_test_df: Full test DataFrame.
        X_orig_df: Original data DataFrame for injection.
        y_orig_arr: Original data target array.
        feat_cols: List of feature column names.

    Returns:
        Tuple of (oof_preds, test_preds, cv_score, last_model).
    """
    kfold = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    oof_preds = np.zeros((len(X), 3))
    test_preds = np.zeros((len(X_test_df), 3))
    fold_scores = []
    last_model = None

    print(f"\n{'=' * 50}")
    print(f"  Training {model_name} ({N_FOLDS}-fold CV)")
    print(f"{'=' * 50}")

    for fold, (train_idx, val_idx) in enumerate(kfold.split(X, y)):
        X_tr = X.iloc[train_idx][feat_cols].copy()
        y_tr = y[train_idx].copy()
        X_va = X.iloc[val_idx][feat_cols].copy()
        X_te = X_test_df[feat_cols].copy()

        # Inject original data into training fold
        X_orig_fold = X_orig_df[feat_cols].copy()
        X_tr = pd.concat([X_tr, X_orig_fold], axis=0).reset_index(drop=True)
        y_tr = np.concatenate([y_tr, y_orig_arr])

        # In-fold target encoding and supervised binning
        X_tr, X_va, X_te = apply_fold_features(X_tr, X_va, X_te, y_tr)

        # Final feature list after fold features
        fold_features = [
            c for c in X_tr.columns
            if c not in (set(CAT_COLS) | {"id", TARGET})
        ]

        # Train
        model = train_fn(X_tr[fold_features], y_tr, X_va[fold_features], y[val_idx])

        # Predict
        oof_preds[val_idx] = model.predict_proba(X_va[fold_features])
        test_preds += model.predict_proba(X_te[fold_features]) / N_FOLDS

        score = balanced_accuracy_score(y[val_idx], oof_preds[val_idx].argmax(axis=1))
        fold_scores.append(score)
        print(f"  Fold {fold + 1}/{N_FOLDS}: BA = {score:.5f}")

        last_model = model
        del X_tr, X_va, X_te
        gc.collect()

    cv_score = balanced_accuracy_score(y, oof_preds.argmax(axis=1))
    print(f"  {model_name} CV: {cv_score:.5f} (std: {np.std(fold_scores):.5f})")

    return oof_preds, test_preds, cv_score, last_model

In [ ]:
print("Training XGBoost...")
oof_xgb, pred_xgb, cv_xgb, model_xgb = run_model_cv(
    "XGBoost", train_xgb_fold, train, y_all, test, orig, y_orig, feature_cols,
)

In [ ]:
print("Training LightGBM...")
oof_lgb, pred_lgb, cv_lgb, model_lgb = run_model_cv(
    "LightGBM", train_lgb_fold, train, y_all, test, orig, y_orig, feature_cols,
)

In [ ]:
print("Training CatBoost...")
oof_cat, pred_cat, cv_cat, model_cat = run_model_cv(
    "CatBoost", train_cat_fold, train, y_all, test, orig, y_orig, feature_cols,
)

In [ ]:
print("\nAll models trained.")
print(f"  XGBoost  CV: {cv_xgb:.5f}")
print(f"  LightGBM CV: {cv_lgb:.5f}")
print(f"  CatBoost CV: {cv_cat:.5f}")

<div style="padding: 12px 20px; border-left: 4px solid #95a5a6; background-color: #f8f9fa; margin: 10px 0;">
<h3 style="color: #2c3e50; margin: 0;">6.1 Feature Importance (Best Model)</h3>
</div>

In [ ]:
# Use XGBoost feature importance as representative
importances = model_xgb.feature_importances_
feat_names = model_xgb.feature_names_in_

imp_df = pd.DataFrame({
    "feature": feat_names,
    "importance": importances,
}).sort_values("importance", ascending=False).head(25)

fig, ax = plt.subplots(figsize=(10, 8))
sns.barplot(
    data=imp_df, x="importance", y="feature",
    palette="viridis", ax=ax, edgecolor="white", linewidth=0.5,
)
ax.set_title("Top 25 Feature Importances (XGBoost)", fontweight="bold")
ax.set_xlabel("Importance (Gain)")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

<div style="padding: 20px; border-left: 5px solid #8e44ad; background: linear-gradient(to right, #f4ecf7, #ffffff); margin: 10px 0;">
<h2 style="color: #2c3e50; margin: 0;">7. Ensemble: Hill Climbing + Threshold Optimization</h2>
<p style="color: #7f8c8d; margin: 5px 0 0 0;">Hill climbing weights, log-bias tuning, differential evolution thresholds</p>
</div>

<div style="padding: 15px 20px; background-color: #eef5fc; border-left: 4px solid #2980b9; border-radius: 4px; margin: 10px 0;">
<p style="margin: 0 0 10px 0;">Three-stage post-processing:</p>
<ol style="margin: 0; padding-left: 20px;">
<li>Hill climbing to find optimal blend weights</li>
<li>Log-space bias tuning (grid search with decreasing steps)</li>
<li>Differential evolution for class-specific threshold multipliers</li>
</ol>
<p style="margin: 10px 0 0 0;">The method that achieves the best CV score is selected for final submission.</p>
</div>

In [ ]:
oof_dict = {"XGBoost": oof_xgb, "LightGBM": oof_lgb, "CatBoost": oof_cat}
pred_dict = {"XGBoost": pred_xgb, "LightGBM": pred_lgb, "CatBoost": pred_cat}
available_models = list(oof_dict.keys())


def hill_climbing_ensemble(oof_dict, y_true, n_iterations=10000, seed=42):
    """Find optimal blend weights via stochastic hill climbing.

    Starts from uniform weights and makes small random perturbations.
    Keeps the perturbation if balanced accuracy improves.

    Args:
        oof_dict: Dict mapping model name to OOF predictions (N, 3).
        y_true: Ground truth labels (numeric).
        n_iterations: Number of search iterations.
        seed: Random state.

    Returns:
        Tuple of (weights dict, best score).
    """
    rng = np.random.RandomState(seed)
    names = list(oof_dict.keys())
    oofs = [oof_dict[n] for n in names]
    n_models = len(oofs)

    weights = np.ones(n_models) / n_models
    blend = sum(w * o for w, o in zip(weights, oofs))
    best_score = balanced_accuracy_score(y_true, blend.argmax(axis=1))
    best_weights = weights.copy()

    for i in range(n_iterations):
        lr = max(0.003, 0.1 * (1 - i / n_iterations))
        new_weights = best_weights + rng.randn(n_models) * lr
        new_weights = np.clip(new_weights, 0, None)
        new_weights /= new_weights.sum()

        blend = sum(w * o for w, o in zip(new_weights, oofs))
        score = balanced_accuracy_score(y_true, blend.argmax(axis=1))

        if score > best_score:
            best_score = score
            best_weights = new_weights.copy()

    return {n: float(w) for n, w in zip(names, best_weights)}, best_score


print("Running hill climbing ensemble (10000 iterations)...")
weights, ensemble_cv = hill_climbing_ensemble(oof_dict, y_all)
print(f"Hill Climbing Ensemble CV: {ensemble_cv:.5f}")
print(f"Weights: {weights}")

In [ ]:
# Blended predictions using hill climbing weights
oof_blend = sum(weights[n] * oof_dict[n] for n in available_models)
test_blend = sum(weights[n] * pred_dict[n] for n in available_models)

baseline_cv = balanced_accuracy_score(y_all, oof_blend.argmax(axis=1))
print(f"Ensemble before thresholds: {baseline_cv:.5f}")

<div style="padding: 12px 20px; border-left: 4px solid #95a5a6; background-color: #f8f9fa; margin: 10px 0;">
<h3 style="color: #2c3e50; margin: 0;">7.1 Log-Space Bias Tuning</h3>
</div>

In [ ]:
def log_bias_tuning(oof_blend, y_true, n_rounds=5):
    """Grid search for class-specific log-space biases.

    Iteratively refines 3 class biases, starting with a coarse grid
    and narrowing with each round.

    Args:
        oof_blend: Blended OOF predictions (N, 3).
        y_true: Ground truth labels (numeric).
        n_rounds: Number of refinement rounds.

    Returns:
        Tuple of (best biases array, best score).
    """
    best_biases = np.zeros(3)
    best_score = balanced_accuracy_score(y_true, oof_blend.argmax(axis=1))

    step = 1.0
    for round_i in range(n_rounds):
        improved = True
        while improved:
            improved = False
            for cls_idx in range(3):
                for delta in [-step, step]:
                    trial = best_biases.copy()
                    trial[cls_idx] += delta
                    adjusted = np.log(oof_blend + 1e-15) + trial
                    preds = adjusted.argmax(axis=1)
                    score = balanced_accuracy_score(y_true, preds)
                    if score > best_score:
                        best_score = score
                        best_biases = trial
                        improved = True
        step *= 0.5

    return best_biases, best_score


log_biases, log_cv = log_bias_tuning(oof_blend, y_all)
print(f"Log-space bias tuning CV: {log_cv:.5f}")
print(f"Biases: {np.round(log_biases, 4)}")

<div style="padding: 12px 20px; border-left: 4px solid #95a5a6; background-color: #f8f9fa; margin: 10px 0;">
<h3 style="color: #2c3e50; margin: 0;">7.2 Differential Evolution Thresholds</h3>
</div>

In [ ]:
def neg_ba_thresholds(thresholds):
    """Negative balanced accuracy for scipy optimization."""
    return -balanced_accuracy_score(y_all, (oof_blend * thresholds).argmax(axis=1))


print("Running differential evolution (maxiter=2000, popsize=30)...")
de_result = differential_evolution(
    neg_ba_thresholds,
    bounds=[(0.3, 4.0), (0.3, 4.0), (0.3, 4.0)],
    seed=SEED,
    maxiter=2000,
    popsize=30,
    tol=1e-12,
)
de_thresholds = de_result.x
de_cv = -de_result.fun

print(f"DE thresholds: {np.round(de_thresholds, 4)}")
print(f"DE threshold CV: {de_cv:.5f}")

<div style="padding: 12px 20px; border-left: 4px solid #95a5a6; background-color: #f8f9fa; margin: 10px 0;">
<h3 style="color: #2c3e50; margin: 0;">7.3 Pick Best Method</h3>
</div>

In [ ]:
methods = {
    "hill_climb_only": (baseline_cv, "argmax", None),
    "log_bias": (log_cv, "log_bias", log_biases),
    "de_thresholds": (de_cv, "de_thresholds", de_thresholds),
}

best_method_name = max(methods, key=lambda k: methods[k][0])
best_cv, best_type, best_params = methods[best_method_name]

print(f"\nMethod comparison:")
for name, (cv, _, _) in methods.items():
    marker = " <-- BEST" if name == best_method_name else ""
    print(f"  {name:20s}  CV = {cv:.5f}{marker}")

# Apply best method to test predictions
if best_type == "argmax":
    final_test_labels_idx = test_blend.argmax(axis=1)
    final_oof_labels_idx = oof_blend.argmax(axis=1)
elif best_type == "log_bias":
    final_test_labels_idx = (np.log(test_blend + 1e-15) + best_params).argmax(axis=1)
    final_oof_labels_idx = (np.log(oof_blend + 1e-15) + best_params).argmax(axis=1)
elif best_type == "de_thresholds":
    final_test_labels_idx = (test_blend * best_params).argmax(axis=1)
    final_oof_labels_idx = (oof_blend * best_params).argmax(axis=1)

final_cv = balanced_accuracy_score(y_all, final_oof_labels_idx)
print(f"\nFinal ensemble CV: {final_cv:.5f}")

<div style="padding: 12px 20px; border-left: 4px solid #95a5a6; background-color: #f8f9fa; margin: 10px 0;">
<h3 style="color: #2c3e50; margin: 0;">7.4 Ensemble Diagnostics</h3>
</div>

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Confusion matrix
cm = confusion_matrix(y_all, final_oof_labels_idx)
sns.heatmap(
    cm, annot=True, fmt=",d", cmap="Blues", ax=axes[0],
    xticklabels=CLASS_ORDER, yticklabels=CLASS_ORDER,
    linewidths=0.5, cbar_kws={"shrink": 0.8},
)
axes[0].set_title(f"Confusion Matrix (CV={final_cv:.5f})", fontweight="bold")
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("True")

# 2. Per-class recall
recalls = []
for ci, cls in enumerate(CLASS_ORDER):
    mask = y_all == ci
    recall = (final_oof_labels_idx[mask] == ci).mean()
    recalls.append(recall)

ax = axes[1]
bars = ax.bar(CLASS_ORDER, recalls, color=CLASS_PALETTE, edgecolor="white", linewidth=1.5)
for bar, r in zip(bars, recalls):
    ax.text(bar.get_x() + bar.get_width() / 2, r + 0.003, f"{r:.4f}",
            ha="center", fontweight="bold")
ax.set_ylabel("Recall")
ax.set_title("Per-Class Recall", fontweight="bold")
ax.set_ylim(min(recalls) - 0.02, 1.0)
ax.axhline(y=final_cv, color="gray", linestyle="--", alpha=0.5, label="Mean BA")
ax.legend()

# 3. Model weights
ax = axes[2]
model_colors = ["#3498db", "#2ecc71", "#e74c3c"]
w_names = list(weights.keys())
w_values = [weights[n] for n in w_names]
bars = ax.barh(w_names, w_values, color=model_colors, edgecolor="white", linewidth=1.5)
for bar, w in zip(bars, w_values):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height() / 2,
            f"{w:.3f}", va="center", fontweight="bold")
ax.set_title("Ensemble Weights (Hill Climbing)", fontweight="bold")
ax.set_xlabel("Weight")

plt.tight_layout()
plt.show()

for ci, cls in enumerate(CLASS_ORDER):
    mask = y_all == ci
    print(f"  {cls:8s} recall: {recalls[ci]:.4f}  (n={mask.sum():,})")

<div style="padding: 20px; border-left: 5px solid #2980b9; background: linear-gradient(to right, #eef5fc, #ffffff); margin: 10px 0;">
<h2 style="color: #2c3e50; margin: 0;">8. Submission</h2>
<p style="color: #7f8c8d; margin: 5px 0 0 0;">Generate and save the final prediction file</p>
</div>

In [ ]:
labels = le.inverse_transform(final_test_labels_idx)
submission = pd.DataFrame({"id": test_ids, "Irrigation_Need": labels})
submission.to_csv("submission.csv", index=False)

print(f"Submission saved: submission.csv ({len(submission)} rows)")
print(f"\nClass distribution:")
print(submission["Irrigation_Need"].value_counts().to_string())
print(f"\nFinal CV: {final_cv:.5f}")
print(f"Best method: {best_method_name}")
print(f"\nFile path: {Path('submission.csv').resolve()}")